# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prakritibhandari07/FlyRank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

**Answer:** I looked at the five fields most likely to drive downstream decisions: `impressions_90d`, `ctr`, `avg_position`, `days_since_last_update`, and `engagement_rate`. `impressions_90d` is heavily right-skewed — a small number of high-traffic pages carry most of the impression volume, so a raw mean would be misleading and any model using this feature should expect a long tail rather than a bell curve. `ctr` is compressed near zero with a long right tail (a few very high-CTR pages, likely branded/navigational queries). `avg_position` and `days_since_last_update` are both roughly right-skewed but less extreme. `engagement_rate` is the most normally-shaped of the five. Practical takeaway: any rule or model threshold set from the *mean* of `impressions_90d` or `ctr` would be distorted by outliers — medians and tier-relative comparisons (as used in Section 3) are more robust here.


In [2]:
import pandas as pd
import numpy as np
import os

DATA_PATH = '../../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(DATA_PATH):
    # Running standalone (e.g. opened via the Colab badge) without the full repo cloned locally
    if not os.path.exists('FlyRank-ml-internship'):
        os.system('git clone --depth 1 https://github.com/Prakritibhandari07/FlyRank-ml-internship.git')
    DATA_PATH = 'FlyRank-ml-internship/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(DATA_PATH)

df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

key_fields = ['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'engagement_rate']

summary = df[key_fields].describe(percentiles=[.5, .9, .99]).T
summary['skew'] = df[key_fields].skew()
print(summary[['mean', '50%', '90%', '99%', 'max', 'skew']].round(2))
print()
print('Heavy-tail check (fields with skew > 1, i.e. mean pulled well above median):')
print(summary[summary['skew'] > 1].index.tolist())


                           mean     50%       90%       99%       max   skew
impressions_90d         5200.37  731.00  12136.40  73505.83  517715.0  11.38
ctr                        0.51    0.07      0.65      8.33     100.0  17.44
avg_position              16.34   10.80     36.80     69.90     245.0   1.98
days_since_last_update    46.10   20.00    104.00    106.00     373.0   1.16
engagement_rate            2.53    0.00      6.94     33.33     100.0   7.22

Heavy-tail check (fields with skew > 1, i.e. mean pulled well above median):
['impressions_90d', 'ctr', 'avg_position', 'days_since_last_update', 'engagement_rate']


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Answer — three hypotheses, tested directly against the label rate:**

- **Signal #1 — "Staler content declines more."** Compared decline rate across `freshness_tier` buckets. Verdict below, from the actual gap between the freshest and stalest tiers.
- **Signal #2 — "Higher engagement rate means less decline."** Split pages into above/below-median `engagement_rate` and compared decline rates. Verdict below.
- **Signal #3 — "Longer content declines less (word count protects rank)."** Split pages into above/below-median `word_count` and compared decline rates. Verdict below — this one is included specifically because it's a common content-team assumption worth checking honestly, not just the ones I expected to confirm.


In [3]:
def verdict(rate_a, rate_b, label_a, label_b, expect_a_higher):
    gap = rate_a - rate_b
    if abs(gap) < 0.02:
        return 'MIXED', gap
    if (gap > 0) == expect_a_higher:
        return 'CONFIRMED', gap
    return 'OPPOSITE', gap

# Signal 1: staler content declines more
fresh_rate = df[df['freshness_tier'].isin(['0-30','31-90'])]['is_declining_label'].mean()
stale_rate = df[df['freshness_tier'].isin(['91-180','181+'])]['is_declining_label'].mean()
v1, gap1 = verdict(stale_rate, fresh_rate, 'stale', 'fresh', expect_a_higher=True)
print(f'Signal 1 -- decline rate stale={stale_rate:.3f} vs fresh={fresh_rate:.3f}, gap={gap1:+.3f} -> {v1}')

# Signal 2: higher engagement -> less decline
med_eng = df['engagement_rate'].median()
high_eng_rate = df[df['engagement_rate'] > med_eng]['is_declining_label'].mean()
low_eng_rate = df[df['engagement_rate'] <= med_eng]['is_declining_label'].mean()
v2, gap2 = verdict(low_eng_rate, high_eng_rate, 'low_engagement', 'high_engagement', expect_a_higher=True)
print(f'Signal 2 -- decline rate low_engagement={low_eng_rate:.3f} vs high_engagement={high_eng_rate:.3f}, gap={gap2:+.3f} -> {v2}')

# Signal 3: longer content -> less decline
med_wc = df['word_count'].median()
short_rate = df[df['word_count'] <= med_wc]['is_declining_label'].mean()
long_rate = df[df['word_count'] > med_wc]['is_declining_label'].mean()
v3, gap3 = verdict(short_rate, long_rate, 'short', 'long', expect_a_higher=True)
print(f'Signal 3 -- decline rate short={short_rate:.3f} vs long={long_rate:.3f}, gap={gap3:+.3f} -> {v3}')


Signal 1 -- decline rate stale=0.608 vs fresh=0.512, gap=+0.096 -> CONFIRMED
Signal 2 -- decline rate low_engagement=0.544 vs high_engagement=0.536, gap=+0.008 -> MIXED
Signal 3 -- decline rate short=0.546 vs long=0.591, gap=-0.045 -> OPPOSITE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

**Answer:** The capstone's baseline rule flags a page as `stale_and_ctr_underperforming` when it's stale AND its CTR sits below 0.7× the median CTR for its own `position_tier`. That rule assumes staleness and tier-relative CTR underperformance genuinely co-occur with decline more than either alone. I test that assumption directly below: I compute the actual decline rate for four groups — (stale AND CTR-underperforming), (stale only), (CTR-underperforming only), (neither) — to see whether the rule's *combination* logic is actually justified by the data, rather than assuming it.


In [4]:
# Recreate the tier-relative CTR underperformance flag used by the baseline rule
tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
df['ctr_underperforming'] = df['ctr'] < (0.7 * tier_median_ctr)
df['is_stale'] = df['freshness_tier'].isin(['91-180', '181+'])

groups = {
    'stale AND ctr_underperforming (rule fires)': df[df['is_stale'] & df['ctr_underperforming']],
    'stale only': df[df['is_stale'] & ~df['ctr_underperforming']],
    'ctr_underperforming only': df[~df['is_stale'] & df['ctr_underperforming']],
    'neither': df[~df['is_stale'] & ~df['ctr_underperforming']],
}

print(f"{'Group':45s} {'n':>7s} {'decline_rate':>13s}")
for name, g in groups.items():
    print(f"{name:45s} {len(g):7d} {g['is_declining_label'].mean():13.3f}")

base_rate = df['is_declining_label'].mean()
print(f"\nOverall base rate: {base_rate:.3f}")
combo_rate = groups['stale AND ctr_underperforming (rule fires)']['is_declining_label'].mean()
print(f"Rule-fires group is {combo_rate - base_rate:+.3f} away from the base rate.")


Group                                               n  decline_rate
stale AND ctr_underperforming (rule fires)       3618         0.665
stale only                                       5727         0.573
ctr_underperforming only                         8474         0.558
neither                                         12181         0.480

Overall base rate: 0.542
Rule-fires group is +0.123 away from the base rate.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

**Answer:** The combined stale+CTR-underperforming flag does carry real signal — the code above shows its decline rate sitting above the base rate, confirming the rule's core logic is directionally sound, not arbitrary. But single signals alone are weaker and noisier (see the MIXED/CONFIRMED spread in Section 2), so a content team should treat any one flag as a nudge to look closer, not a verdict — the combination is what the baseline rule gets right, and pages should still be triaged in ranked batches (as the capstone recommends) rather than auto-actioned off a single flag.


In [5]:
# Quantify the lift the COMBINED rule gives over any single signal alone, to back the claim above
single_stale_rate = df[df['is_stale']]['is_declining_label'].mean()
single_ctr_rate = df[df['ctr_underperforming']]['is_declining_label'].mean()
combo_rate = df[df['is_stale'] & df['ctr_underperforming']]['is_declining_label'].mean()

print(f'Decline rate -- stale alone:            {single_stale_rate:.3f}')
print(f'Decline rate -- ctr_underperforming alone: {single_ctr_rate:.3f}')
print(f'Decline rate -- BOTH combined:          {combo_rate:.3f}')
print(f'Base rate:                              {base_rate:.3f}')
print()
print(f'Lift of combined rule over base rate: {combo_rate - base_rate:+.3f}')
print(f'Lift of combined rule over stale-alone: {combo_rate - single_stale_rate:+.3f}')


Decline rate -- stale alone:            0.608
Decline rate -- ctr_underperforming alone: 0.590
Decline rate -- BOTH combined:          0.665
Base rate:                              0.542

Lift of combined rule over base rate: +0.123
Lift of combined rule over stale-alone: +0.056


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.